This notebook removes filters samples with a large amount of outliers, and converts the data to a long format of gene-individual pairs.

This notebook should take about 30 seconds to run.

## Setup

In [1]:
library(data.table)

In [2]:
d <- fread('mage_expression-tpm_residuals.csv')

## Filter samples with a suspicious amount of high z scores (an amount > 1.5*IQR)
This is meant to remove samples with high expression values which may be due to technical errors.

In [3]:
OUTLIER_Z_THRESH <- 3 # A Z-score more extreme than this is defined as an "outlier".

high_zs_per_sample <- rowSums(abs(d[,-c('sample_id')]) > OUTLIER_Z_THRESH)
too_many_high_z_thresh <- quantile(high_zs_per_sample,c(.25,.75)) |> diff()*1.5
d <- d[high_zs_per_sample < too_many_high_z_thresh]

In [4]:
paste('Samples removed: ', sum(high_zs_per_sample > too_many_high_z_thresh))

[1] "Samples removed:  191"

## Convert to long format (3 columns: `gene`, `sample_id`, `z`)

In [5]:
d <- melt(d,
  id.vars='sample_id',
  variable.name='gene_id',
  value.name='z'
)

In [6]:
head(d)

sample_id,gene_id,z
<chr>,<fct>,<dbl>
NA06985,ENSG00000002745.13,-1.20296198
NA07000,ENSG00000002745.13,-0.84021613
NA11919,ENSG00000002745.13,-0.17164761
NA11933,ENSG00000002745.13,0.05810866
NA12003,ENSG00000002745.13,0.25246033
NA12286,ENSG00000002745.13,0.66793385


## Filter to genes with at least one outlier

In [7]:
genes_w_high_z <- unique(d[abs(z) > OUTLIER_Z_THRESH, gene_id])
d <- d[gene_id %in% genes_w_high_z]

## Write

In [8]:
setnames(d, c('indiv','gene','z'))
fwrite(d,'indiv_gene_z.tsv', sep='\t')
system('gcloud storage cp indiv_gene_z.tsv $WORKSPACE_BUCKET/data/derived/')